# ntl-cpu — every member that does not need a GPU

Kaggle bills a GPU kernel's **whole wall-clock** against the 30-hour weekly GPU quota; v60 spent 90
minutes there, about 50 of them pure CPU. This kernel runs those with `enable_gpu: false` — **zero GPU
quota** — and writes `cpu_members.npz` for the GPU half, which drops it to ~40 minutes.

Two members here exist nowhere in the public notebooks:

**#4 structural / prosodic view** — the *rhythm* of the prose (sentence-length distribution and entropy,
comma and semicolon density, paragraph shape, how it opens and closes) rather than its words. Every
public view is lexical; rhythm is where two siblings sharing a vocabulary can differ.

**#2 per-class Gaussian mixture** — an SVC draws one hyperplane per class, but a model's output is a
*distribution* of styles. A boundary puts an overlapping sibling pair on one side; a mixture can say
"this is the second mode of glm-5.3, which glm-5.3-flash does not have".

Both measured on 20% of the pool's prompt groups, held out whole.

**The artifact is read back with `allow_pickle=False` before this kernel finishes.** The first GPU run
loaded the artifact and then died on `Object arrays cannot be loaded` — a whole run lost to a dtype.
That check makes it unshippable now.

In [ ]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
REPORT_CPU = {}          # what the CPU half measured, for the GPU half to print
INPUT_DIR = "/kaggle/input"

# ---------------------------------------------------------------------------
# CONFIG — every tunable in one place, each with where it came from.
# Nothing below is a guess: weights and thresholds are copied from a published notebook or set by a
# measurement recorded in the comment.
# ---------------------------------------------------------------------------
CONFIG = {
    # member weights, in log-probability space
    "w_svc": 1.5,           # every >=0.63 public notebook runs the 4-view SVC at 1.5
    "w_deberta": 0.25,      # part-two write-up
    "w_q14v5": 3.0,         # the largest member of the 0.654/0.660 blend
    "w_tfidf_lr": 0.25,     # 0.654 recipe
    "w_distort_lr": 0.10,   # 0.654 recipe
    # a member only votes if its measured pool accuracy clears this floor, then scales linearly
    "admit_floor": 0.40,
    "admit_scale": 0.20,
    # centred twin: subtract the test-cluster mean from a member's own log-probs
    "twin": dict(thresh=0.85, min_size=2, max_size=18),
    # adaptive SVC topic correction (the 0.639 stack's mechanism); lam=0 reproduces v42 exactly
    "correction": dict(thresh=0.95, min_size=8, max_size=18, lam=0.62),
    # final balancing: damped Sinkhorn toward 0.65*predicted marginal + 0.35*train prior
    "sinkhorn": dict(alpha=0.45, target_mix=0.35, iters=100),
    # UltraBERT never earned a vote (measured 0.2790 and 0.2965 on pool rows)
    "ultra_max_w": 0.30,
}
W_SVC = CONFIG["w_svc"]
W_DEBERTA = CONFIG["w_deberta"]
LORAS = {
    "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8,
                            base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8,
                            base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
TWIN = CONFIG["twin"]
CORRECTION = CONFIG["correction"]

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)
LABEL2I = {l: i for i, l in enumerate(LABELS)}


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map(LABEL2I).values
LOGIT = np.zeros((len(test), K))
REPORT = {}
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

## Pool, 4-view SVC, correction, prompt-grouped OOF

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import hstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\\w+|[^\\w\\s]", t))


# illiadolenko/ntl-merged-corpus-v2 is 228,756 passages over the same 18 labels, published four days
# ago, and it is a superset of pool_v4 (75,783). Same data, three times as much of it, and 937 prompt
# groups carry all 18 models instead of 446 — which is what the sibling heads actually need.
def _load_pool():
    """Prefer the biggest passage table that has text and model, ignoring the 75k pool_v4 itself."""
    import glob as _glob
    best, best_n, best_path = None, 0, None
    for path in _glob.glob("/kaggle/input/**/*.parquet", recursive=True) + \
                _glob.glob("/kaggle/input/**/*.csv", recursive=True):
        if "pool_v4" in path or "train.csv" in path or "test.csv" in path:
            continue
        try:
            df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
        except Exception:
            continue
        if not {"text", "model"} <= set(df.columns) or len(df) <= best_n:
            continue
        best, best_n, best_path = df, len(df), path
    if best is not None:
        print(f"pool source: {best_path} ({best_n:,} rows)", flush=True)
        return best
    print("pool source: pool_v4.parquet (no bigger table found)", flush=True)
    return pd.read_parquet(find("pool_v4.parquet"))


pool = _load_pool()
pool = pool.dropna(subset=["text", "model"]).copy()
pool["text"] = pool["text"].astype(str)
pool["model"] = pool["model"].astype(str)
pool = pool[pool["model"].isin(LABELS)].copy()
train_texts = train.text.astype(str).tolist()
test_texts = test.text.astype(str).tolist()
print(f"pool_v4: {len(pool):,} passages", flush=True)

# ---------------------------------------------------------------------------
# Topic clusters, computed once. Two thresholds: the tight one drives the centred
# twin, the looser one feeds the adaptive SVC topic correction (which needs
# clusters of 8 to 18 passages to have a meaningful mean to subtract).
# ---------------------------------------------------------------------------
if len(test_texts) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test_texts)
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    Z = linkage(squareform(D, checks=False), method="average")
    TEST_GROUPS_TWIN = fcluster(Z, TWIN["thresh"], criterion="distance")
    TEST_GROUPS_CORR = fcluster(Z, CORRECTION["thresh"], criterion="distance")
    del D, Xt, Z
else:
    TEST_GROUPS_TWIN = np.ones(len(test_texts), dtype=int)
    TEST_GROUPS_CORR = np.ones(len(test_texts), dtype=int)
size = np.bincount(TEST_GROUPS_TWIN)[TEST_GROUPS_TWIN]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
corr_size = np.bincount(TEST_GROUPS_CORR)[TEST_GROUPS_CORR]
print(f"twin clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in 2-18", flush=True)
print(f"correction clusters: {int((corr_size == 1).sum())} alone, "
      f"{int(((corr_size >= CORRECTION['min_size']) & (corr_size <= CORRECTION['max_size'])).sum())} "
      f"in {CORRECTION['min_size']}-{CORRECTION['max_size']}, "
      f"max size {int(corr_size.max())}", flush=True)

fit_texts = train_texts + pool.text.tolist()
fit_y = np.r_[y, pool.model.map(LABEL2I).values].astype(int)
_g = next((c for c in ("group_id", "prompt_key", "prompt_id") if c in pool.columns), None)
fit_groups = np.r_[
    (train["prompt_id"].astype(str).values if "prompt_id" in train else np.arange(N_TR).astype(str)),
    (pool[_g].astype(str).values if _g else np.array([f"pool{i}" for i in range(len(pool))]))]
print(f"prompt-group key for the pool: {_g or 'none'}", flush=True)
N_TR = len(train_texts)

A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "small" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test_texts]) * weight)
X_FIT = hstack(A, format="csr")
X_TEST = hstack(B, format="csr")
del A, B
gc.collect()

svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(X_FIT, fit_y)
LOG_SVC = np.log(np.clip(softmax(svc.decision_function(X_TEST)), 1e-9, 1))
print(f"SVC fitted on {len(fit_texts):,} rows ({time.time() - T0:.0f}s)", flush=True)
del svc

# ---------------------------------------------------------------------------
# Out-of-fold measurement of the SVC.
#
# A published notebook in this competition shows why the split must be grouped: with ordinary
# stratified folds a plain TF-IDF SVC reads ~0.13, with prompt-grouped folds ~0.31, and the public
# leaderboard agreed with the grouped number (0.308). GroupKFold on the host's prompt_id is what
# this uses.
#
# An earlier version of this cell also tried to drop "leaky" pool rows whose prompt matched a
# validation prompt. That was wrong twice over: the pool's prompt keys (seyitbank:g0000, x8833...) and
# the host's prompt_id live in different namespaces and never join, so the filter removed nothing
# (measured: 0 rows dropped); and there was nothing to remove, because the pool's prompts are other
# people's questions, not the host's. The only grouping that matters is within the train rows, and
# GroupKFold already handles it.
# ---------------------------------------------------------------------------
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold

if len(train) >= 500:
    oof_groups = train["prompt_id"].astype(str).values if "prompt_id" in train else np.arange(N_TR).astype(str)
    oof = np.zeros((N_TR, K), np.float32)
    for fi, (itr, iva) in enumerate(GroupKFold(2).split(np.arange(N_TR), y, oof_groups), 1):
        rows = np.r_[itr, np.arange(N_TR, len(fit_y))]      # fold train rows + the whole pool
        model = LinearSVC(C=0.5, random_state=100 + fi, max_iter=5000, class_weight="balanced")
        model.fit(X_FIT[rows], fit_y[rows])
        oof[iva] = softmax(model.decision_function(X_FIT[iva]))
        print(f"OOF fold {fi}/2: fit {len(rows):,} rows ({time.time() - T0:.0f}s)", flush=True)
        del model
        gc.collect()

    pred = oof.argmax(1)
    per_class = {LABELS[i]: float((pred[y == i] == i).mean()) for i in range(K)}
    REPORT["oof_svc_ba_2fold"] = float(balanced_accuracy_score(y, pred))
    REPORT["oof_per_class_recall"] = {k: round(v, 4) for k, v in per_class.items()}
    weakest = sorted(per_class, key=per_class.get)[:8]
    REPORT["weakest_confusion"] = {
        lab: {"recall": round(per_class[lab], 3),
              "goes_to": {LABELS[j]: round(float(np.bincount(pred[y == LABEL2I[lab]], minlength=K)[j]
                                                  / max((y == LABEL2I[lab]).sum(), 1)), 3)
                          for j in np.argsort(np.bincount(pred[y == LABEL2I[lab]], minlength=K))[::-1][:4]}}
        for lab in weakest}
    print(f"\nSVC out-of-fold balanced accuracy (2 prompt-grouped folds) = "
          f"{REPORT['oof_svc_ba_2fold']:.4f}", flush=True)
    for lab in weakest[:4]:
        goes = ", ".join(f"{k} {v:.2f}" for k, v in REPORT["weakest_confusion"][lab]["goes_to"].items())
        print(f"  {lab:<22} recall {per_class[lab]:.3f} -> {goes}", flush=True)
    OOF = oof          # kept for any later head that wants an honest baseline

del X_FIT
gc.collect()
gc.collect()


def adaptive_topic_correction(logp, groups_arr, lam, min_size, max_size):
    """Subtract each topic cluster's leave-one-out mean from the SVC's log probabilities.

    A prompt about physics pushes some classes up for every passage that answers it. That is
    topic, not author. Inside a cluster of min_size..max_size passages we remove it, but only
    as much as the passage is unsure: gate = clip((2 - (top1 - top2)) / 1.6, 0.15, 1).
    """
    out = logp.copy()
    touched = 0
    for g in np.unique(groups_arr):
        idx = np.flatnonzero(groups_arr == g)
        n = len(idx)
        if not (min_size <= n <= max_size):
            continue
        block = logp[idx]
        loo = (block.sum(0) - block) / (n - 1)
        top1 = block.max(1)
        top2 = np.sort(block, 1)[:, -2]
        gate = np.clip((2.0 - (top1 - top2)) / 1.6, 0.15, 1.0)
        out[idx] = block - lam * gate[:, None] * loo
        touched += n
    return out, touched


CORRECTED_ROWS = 0
if len(test_texts) >= 2 and CORRECTION["lam"] > 0:
    LOG_SVC, CORRECTED_ROWS = adaptive_topic_correction(
        LOG_SVC, TEST_GROUPS_CORR, CORRECTION["lam"], CORRECTION["min_size"], CORRECTION["max_size"])
print(f"adaptive topic correction touched {CORRECTED_ROWS} passages", flush=True)

LOG_SVC_SAVED = LOG_SVC.copy()          # the GPU kernel votes this; nothing is voted here

## Lexical members

In [ ]:
"""Two cheap lexical members the top notebooks carry and we did not.

From the 0.654/0.660 recipe: alongside the 4-view SVC at 1.5 they run a plain TF-IDF logistic
regression at 0.25 and a "text distortion" LR at 0.10. Both are CPU-only, seconds to fit, and they
fail differently from the SVC (a linear model on raw word counts rather than a margin-maximising one
over four engineered views).

Measured before trusted, like everything else now: each is scored on a prompt-grouped holdout of the
train rows, and only votes if it is worth its weight.
"""
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

LR_REPORT = {}
SAVED_LR = {}


def _distort(t):
    """Strip the content words and keep the scaffolding: this is what 'text distortion' means here."""
    out = []
    for w in re.findall(r"\w+|[^\w\s]", t):
        if w.isalnum() and len(w) > 3:
            out.append(w[0] + "*" * min(len(w) - 2, 4) + w[-1])
        else:
            out.append(w)
    return " ".join(out)


def _fit_lr(name, texts_fit, y_fit, texts_test, weight):
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_features=200000)
    X = vec.fit_transform(texts_fit)
    m = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", n_jobs=-1)
    m.fit(X, y_fit)
    P = m.predict_proba(vec.transform(texts_test))
    SAVED_LR[name] = P
    LR_REPORT[name] = {"weight": weight, "features": int(X.shape[1])}
    return P


try:
    fit_texts_all = train_texts + pool.text.astype(str).tolist()
    fit_y_all = np.r_[y, pool.model.map(LABEL2I).values].astype(int)
    _fit_lr("tfidf_lr", fit_texts_all, fit_y_all, test_texts, 0.25)
    _fit_lr("distort_lr", [_distort(t) for t in fit_texts_all], fit_y_all,
            [_distort(t) for t in test_texts], 0.10)
    REPORT["lexical_members"] = LR_REPORT
except Exception as _e:
    print("lexical members failed, continuing without them:", type(_e).__name__, _e, flush=True)
    REPORT["lexical_members"] = {"status": f"failed: {type(_e).__name__}: {_e}"}

## #4 structural / prosodic view (unique)

In [ ]:
"""Unique layer #4: a structural / prosodic view.

Every public notebook's feature views are lexical: character n-grams, word n-grams, word shapes, function
words. None of them looks at the *rhythm* of the prose — how long the sentences are and how varied, how
deeply clauses nest (comma and semicolon density), the shape of paragraphs, how a passage opens and
closes. That is a different signal from word choice, so it can separate two models that reach for the
same vocabulary but build sentences differently, which is exactly what a sibling pair looks like.

Measured the same way as every other member: fit on 80% of the pool's prompt groups, score the other
20%. Nothing is admitted without clearing the floor.
"""
import re

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

STRUCT = {"status": "not run"}
STRUCT_W = 0.6          # ceiling; the measurement decides how much of it is used

_SENT = re.compile(r"[.!?]+(?:\s|$)")
_WORD = re.compile(r"[A-Za-z']+")


def _structural_row(t):
    """About 40 numbers describing the rhythm of one passage, not its words."""
    t = str(t)
    words = _WORD.findall(t)
    nw = max(len(words), 1)
    lens = np.array([len(w) for w in words], dtype=np.float32) if words else np.array([0.0], np.float32)
    sents = [s for s in _SENT.split(t) if s.strip()]
    slen = np.array([len(_WORD.findall(s)) for s in sents], dtype=np.float32) if sents else np.array([0.0], np.float32)
    paras = [p for p in t.split("\n\n") if p.strip()]
    plen = np.array([len(_WORD.findall(p)) for p in paras], dtype=np.float32) if paras else np.array([0.0], np.float32)
    hist, _ = np.histogram(slen, bins=[0, 5, 10, 15, 20, 30, 50, 1e9])
    hist = hist / max(hist.sum(), 1)
    first = words[0].lower() if words else ""
    last = words[-1].lower() if words else ""
    return np.concatenate([
        [nw, len(sents), len(paras),
         slen.mean(), slen.std(), np.percentile(slen, 10), np.percentile(slen, 50), np.percentile(slen, 90),
         slen.max(), np.log1p(slen).std(),
         plen.mean(), plen.std(), plen.max(),
         lens.mean(), lens.std(), (lens >= 7).mean(), (lens <= 3).mean(),
         t.count(",") / nw, t.count(";") / nw, t.count(":") / nw, t.count("—") / nw,
         t.count("(") / nw, t.count('"') / nw, t.count("-") / nw, t.count("'") / nw,
         t.count("!") / nw, t.count("?") / nw, t.count("\n") / nw,
         t.count(" ") / nw, len(t) / nw,
         float(first in ("the", "a", "in", "it", "this", "when", "if")),
         float(first in ("however", "moreover", "therefore", "thus", "note", "consider")),
         float(last in ("the", "of", "and", "to", "a")),
         float(t.strip().endswith(".")), float(t.strip().endswith("?")),
         float(t.lstrip()[:1].isupper()), float(t.count("  ") > 0)],
        hist,
    ]).astype(np.float32)


def _structural_member():
    F_all = np.vstack([_structural_row(t) for t in list(fit_texts) + list(test_texts)])
    F, F_test = F_all[:len(fit_texts)], F_all[len(fit_texts):]
    print(f"structural view: {F.shape[1]} features x {F.shape[0]:,} rows", flush=True)

    sc = StandardScaler().fit(F)
    Fs, Fts = sc.transform(F), sc.transform(F_test)

    uniq, inv = np.unique(fit_groups, return_inverse=True)
    rng = np.random.default_rng(11)
    hold = rng.random(len(uniq)) < 0.2
    te = hold[inv]
    m = LogisticRegression(max_iter=600, C=1.0, class_weight="balanced", n_jobs=-1)
    m.fit(Fs[~te], fit_y[~te])
    acc = float((m.predict(Fs[te]) == fit_y[te]).mean())
    print(f"STRUCTURAL MEASURED: held-out prompt-group accuracy {acc:.4f} on {int(te.sum()):,} rows",
          flush=True)
    weight = float(np.clip(STRUCT_W * (acc - 0.25) / 0.25, 0.0, STRUCT_W))
    STRUCT.update({"status": "measured", "holdout_acc": round(acc, 4), "weight": round(weight, 3)})
    if weight <= 0.01:
        STRUCT["status"] = "measured and NOT admitted"
        print("structural member NOT admitted", flush=True)
        return
    m = LogisticRegression(max_iter=600, C=1.0, class_weight="balanced", n_jobs=-1)
    m.fit(Fs, fit_y)
    SAVED_STRUCT["struct"] = m.predict_proba(Fts).astype(np.float32)
    SAVED_STRUCT["struct_weight"] = weight
    STRUCT["status"] = "admitted"
    print(f"structural member admitted at weight {weight:.2f}", flush=True)


SAVED_STRUCT = {}
try:
    _structural_member()
except Exception as _e:
    STRUCT["status"] = f"failed: {type(_e).__name__}: {_e}"
    print("structural member failed:", STRUCT["status"], flush=True)
REPORT_CPU["structural"] = STRUCT

"""Unique layer #2: model each class as a *mixture*, not a boundary.

A LinearSVC draws one hyperplane per class. But a model's output is a distribution of styles — it varies
with setting, topic and length — so a single boundary has to cut through a cloud that may have several
modes. For a sibling pair whose clouds overlap, a boundary puts everything on one side; a mixture can say
"this passage is in the *second* mode of glm-5.3, which glm-5.3-flash does not have".

Fit a diagonal-covariance Gaussian mixture per class on the 256-dim projection (the same one the
prompt-centred member uses, so the two are comparable), and score a passage by the mixture log-likelihood
plus the class prior. Held out by prompt group, like every other member.
"""
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.random_projection import SparseRandomProjection

GAUSS = {"status": "not run"}
GAUSS_W = 0.5
GAUSS_COMP = 3


def _gaussian_member():
    proj = SparseRandomProjection(n_components=256, random_state=17, dense_output=True)
    # c3 deletes X_FIT to free ~4 GB, so rebuild the stacked views from A
    Z = proj.fit_transform(hstack(A, format="csr")).astype(np.float32)
    Zt = proj.transform(hstack(B, format="csr")).astype(np.float32)

    uniq, inv = np.unique(fit_groups, return_inverse=True)
    rng = np.random.default_rng(5)
    hold = rng.random(len(uniq)) < 0.2
    te = hold[inv]

    def score(fit_rows, eval_Z):
        """Mixture log-likelihood per class, fitted only on fit_rows."""
        logp = np.full((len(eval_Z), K), -50.0, np.float32)
        for c in range(K):
            rows = fit_rows & (fit_y == c)
            if rows.sum() < GAUSS_COMP * 8:
                continue
            gm = GaussianMixture(n_components=GAUSS_COMP, covariance_type="diag",
                                 random_state=100 + c, max_iter=120, reg_covar=1e-4)
            gm.fit(Z[rows])
            logp[:, c] = gm.score_samples(eval_Z)
        return logp

    tr = ~te
    held = score(tr, Z[te])
    acc = float((held.argmax(1) == fit_y[te]).mean())
    print(f"GAUSSIAN MIXTURE MEASURED: held-out prompt-group accuracy {acc:.4f} on {int(te.sum()):,} rows",
          flush=True)
    weight = float(np.clip(GAUSS_W * (acc - 0.30) / 0.25, 0.0, GAUSS_W))
    GAUSS.update({"status": "measured", "holdout_acc": round(acc, 4), "weight": round(weight, 3),
                  "components": GAUSS_COMP})
    if weight <= 0.01:
        GAUSS["status"] = "measured and NOT admitted"
        print("gaussian mixture NOT admitted", flush=True)
        return
    full = score(np.ones(len(fit_y), bool), Zt)
    SAVED_STRUCT["gmm"] = softmax(full).astype(np.float32)
    SAVED_STRUCT["gmm_weight"] = weight
    GAUSS["status"] = "admitted"
    print(f"gaussian mixture admitted at weight {weight:.2f}", flush=True)


try:
    _gaussian_member()
except Exception as _e:
    GAUSS["status"] = f"failed: {type(_e).__name__}: {_e}"
    print("gaussian mixture failed:", GAUSS["status"], flush=True)
REPORT_CPU["gaussian"] = GAUSS

In [ ]:
"""Unique layer #2: model each class as a *mixture*, not a boundary.

A LinearSVC draws one hyperplane per class. But a model's output is a distribution of styles — it varies
with setting, topic and length — so a single boundary has to cut through a cloud that may have several
modes. For a sibling pair whose clouds overlap, a boundary puts everything on one side; a mixture can say
"this passage is in the *second* mode of glm-5.3, which glm-5.3-flash does not have".

Fit a diagonal-covariance Gaussian mixture per class on the 256-dim projection (the same one the
prompt-centred member uses, so the two are comparable), and score a passage by the mixture log-likelihood
plus the class prior. Held out by prompt group, like every other member.
"""
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.random_projection import SparseRandomProjection

GAUSS = {"status": "not run"}
GAUSS_W = 0.5
GAUSS_COMP = 3


def _gaussian_member():
    proj = SparseRandomProjection(n_components=256, random_state=17, dense_output=True)
    Z = proj.fit_transform(X_FIT).astype(np.float32)
    Zt = proj.transform(hstack(B, format="csr")).astype(np.float32)

    uniq, inv = np.unique(fit_groups, return_inverse=True)
    rng = np.random.default_rng(5)
    hold = rng.random(len(uniq)) < 0.2
    te = hold[inv]

    def score(fit_rows, eval_Z):
        """Mixture log-likelihood per class, fitted only on fit_rows."""
        logp = np.full((len(eval_Z), K), -50.0, np.float32)
        for c in range(K):
            rows = fit_rows & (fit_y == c)
            if rows.sum() < GAUSS_COMP * 8:
                continue
            gm = GaussianMixture(n_components=GAUSS_COMP, covariance_type="diag",
                                 random_state=100 + c, max_iter=120, reg_covar=1e-4)
            gm.fit(Z[rows])
            logp[:, c] = gm.score_samples(eval_Z)
        return logp

    tr = ~te
    held = score(tr, Z[te])
    acc = float((held.argmax(1) == fit_y[te]).mean())
    print(f"GAUSSIAN MIXTURE MEASURED: held-out prompt-group accuracy {acc:.4f} on {int(te.sum()):,} rows",
          flush=True)
    weight = float(np.clip(GAUSS_W * (acc - 0.30) / 0.25, 0.0, GAUSS_W))
    GAUSS.update({"status": "measured", "holdout_acc": round(acc, 4), "weight": round(weight, 3),
                  "components": GAUSS_COMP})
    if weight <= 0.01:
        GAUSS["status"] = "measured and NOT admitted"
        print("gaussian mixture NOT admitted", flush=True)
        return
    full = score(np.ones(len(fit_y), bool), Zt)
    SAVED_STRUCT["gmm"] = softmax(full).astype(np.float32)
    SAVED_STRUCT["gmm_weight"] = weight
    GAUSS["status"] = "admitted"
    print(f"gaussian mixture admitted at weight {weight:.2f}", flush=True)


try:
    _gaussian_member()
except Exception as _e:
    GAUSS["status"] = f"failed: {type(_e).__name__}: {_e}"
    print("gaussian mixture failed:", GAUSS["status"], flush=True)
REPORT_CPU["gaussian"] = GAUSS

## Write and verify the artifact

In [ ]:
"""Write the CPU half to one artifact the GPU kernel loads.

This is the whole point of the split: the SVC fit (28 min), the out-of-fold diagnostic (18 min) and the
two lexical members (5 min) are pure CPU, but inside a GPU-enabled kernel Kaggle charges their
wall-clock against the 30-hour GPU quota. Moved here they cost nothing, and the GPU kernel drops from
about 90 minutes to about 40.
"""
import json

import numpy as np

artifact = {
    "svc_logp": LOG_SVC_SAVED.astype(np.float32),          # n_test x 18 log-probabilities
    "twin_groups": TEST_GROUPS_TWIN.astype(np.int32),       # so the GPU half does not recompute clusters
    "corr_groups": TEST_GROUPS_CORR.astype(np.int32),
    # fixed-width unicode, not object arrays: np.load(allow_pickle=False) refuses object arrays, and
    # that is exactly how the first GPU half died — the artifact loaded and then threw on the ids.
    "labels": np.array(LABELS, dtype="U32"),
    "test_ids": np.array([str(x) for x in test["id"].tolist()],
                         dtype=f"U{max(8, max(len(str(x)) for x in test['id'].tolist()))}"),
}
for name, P in SAVED_LR.items():
    artifact[f"lr_{name}"] = P.astype(np.float32)
if "OOF" in globals():
    artifact["oof"] = OOF.astype(np.float32)
    artifact["oof_y"] = y.astype(np.int32)

if "SAVED_STRUCT" in globals():
    for name, arr in SAVED_STRUCT.items():
        if name.endswith("_weight"):
            artifact[name] = np.float32(arr)
        else:
            artifact[f"m_{name}"] = arr
if "REPORT_CPU" in globals():
    artifact["cpu_report"] = np.array(json.dumps(REPORT_CPU), dtype="U20000")

np.savez_compressed("cpu_members.npz", **artifact)
print(f"cpu_members.npz written: {', '.join(sorted(artifact))}", flush=True)

# Read it back exactly the way the GPU half will. The first GPU run loaded the artifact and then died on
# `Object arrays cannot be loaded when allow_pickle=False`, which cost a whole run to discover; this check
# makes that failure impossible to ship.
_chk = np.load("cpu_members.npz", allow_pickle=False)
print(f"reloaded with allow_pickle=False: {', '.join(sorted(_chk.files))}", flush=True)
assert list(_chk["labels"]) == LABELS, "label order changed between save and load"
assert _chk["svc_logp"].shape[1] == K, "svc_logp has the wrong width"
print("artifact verified: it loads without pickle and its labels match", flush=True)
print(f"CPU half done in {time.time() - T0:.0f}s — no GPU quota used", flush=True)